### Module 2
**1. tools_messages**

**2. chains_reducers**


#### Tools Messages (Tool Calling)

In [2]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [AIMessage(content=f"So you said you were researching about ai research engineers?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right", name="Moiz"))
messages.append(AIMessage(content=f"Great! What specific aspects of ai research engineering are you interested in?", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about those people who works in openai, deepmind, what are their mindset"))

for m in messages:
    m.pretty_print()

================================== Ai Message ==================================
Name: Model

So you said you were researching about ai research engineers?
================================ Human Message =================================
Name: Moiz

Yes, that's right
================================== Ai Message ==================================
Name: Model

Great! What specific aspects of ai research engineering are you interested in?
================================ Human Message =================================

I want to learn about those people who works in openai, deepmind, what are their mindset


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm: ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [7]:
simple_call = llm.invoke("Hi")
simple_call.content

'Hi there! How can I help you today?'

##### Tools are useful whenever you want a model to interact with external system
- Externla systems (e.g., API's) often require a particular input schema or payload, rather than natural language.
- when we bind an API, for example, as a tool we given the model awareness of the required input schema.
- The model will choose to call a tool based upon the natural language input from the user.
- And it will return an output that adheres to the tool's schema.
- Many LLM providers support tool calling and tool calling interface in langchain is simple
- We can simply pass any python function into ChatModel.bind_tools(function)

In [12]:
def deposit_money(name: str, bank_account_no: int, amount: int) -> dict:
    """
    Deposit Money in Bank Account

    Args:
        name: account holder name
        bank_account_no: bank account id
        amount: amount to be deposited

    Returns:
        dict: a dict
    """

    #Business Logic for Bank Deposit
    return {"status": f"Deposit {amount} Sucessful in {name} Account"}

In [13]:
deposit_money(name="Moiz", bank_account_no=1122334455, amount=500)

{'status': 'Deposit 500 Sucessful in Moiz Account'}

In [10]:
llm_with_tools = llm.bind_tools([deposit_money])

In [11]:
llm_with_tools

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001942FFB3D10>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'deposit_money', 'description': 'Deposit Money in Bank Account', 'parameters': {'properties': {'name': {'description

In [14]:
from langchain_core.messages import HumanMessage

call = llm.invoke(
    [HumanMessage(content=f"Deposit 200 in Ahmad Account. His acc number is 5544332211", name="Muhammad")]
)

call

AIMessage(content='I understand you\'d like to deposit money into Ahmad\'s account.\n\nHowever, as an AI, I **cannot perform actual banking transactions or access real-world financial systems.** I don\'t have the ability to move money or interact with bank accounts.\n\nTo deposit 200 into Ahmad\'s account (number 5544332211), you will need to use your bank\'s official channels:\n\n1.  **Your Bank\'s Online Banking Portal or Mobile App:** Log in to your own bank account and look for options like "Transfer," "Pay & Transfer," or "Send Money." You\'ll need Ahmad\'s account number and possibly his bank\'s name.\n2.  **Visit a Bank Branch:** Go to your bank or Ahmad\'s bank (if you know it) and speak to a teller.\n3.  **Use an ATM:** Some ATMs allow cash or check deposits to other accounts, but you\'ll need the account number.\n4.  **Phone Banking:** Call your bank\'s customer service line and follow their instructions for transfers or deposits.\n\nPlease use one of these secure, official m

In [16]:
call = llm_with_tools.invoke(
    [HumanMessage(content=f"Deposit 200 in Ahmad Account. His acc number is 5544332211", name="junaid")]
)

call

AIMessage(content='', additional_kwargs={'function_call': {'name': 'deposit_money', 'arguments': '{"name": "Ahmad", "bank_account_no": 5544332211}'}, '__gemini_function_call_thought_signatures__': {'179924f2-5208-4a6c-8087-f9c320573397': 'CuoDAQw51sdLeAPf9nzD+8o+H2c9xBEMZLSJfvvj/9aCxYuozEqn842x/3pYRNePBg2jiVrv+UgrZAzy4xtFmTVnifhq6FQe5nDYIwwimo6crmBIureKgDoajL908aMn5N4p2LCO0bLCOQbk9aCndPlhoePQYjNHbf2GtFWNfb9C1Y25zwajee3hToEqnVuh/m/4DpVZNvFcZg3tuhTnT8eT1PbMzuzm1zdztCDvlHhMjdwYV9M4izDWDIShL1A+lsEbQM60P0oHH4HvuH1CnZGWYPcWUUjNYV+mNgs1/XB1cz88mi2uhHB+FLySx6BLU95zqyizgwIdUNuMzgJ/DGmkDsPszq3JDR6AAxvdQ4nMqdd1ZIjdkHlHqI/g4oAI+3SMPLvcIhqXHqxIxpSZ5Oxhh3PaxQdjsGWCta0CsgloyG1zK/qQmOazr6An3gEH/yw6Zh9gWLTBtW9G+Q3SK5zjujBNQ03adGgur03H7f9znWH+IZBt5ODCNm12/QkskM9pCg4LTQ20W4hXH24BW6mcoELkocLVCtZvfpRYpfHMwlPokTUBNTb0CIvkMVeajBqUp+h/riHKRc0RVK92aEXkV6zDlZP5hk23rZhfbOVBCg6J9PtE2i5OM+cmMnR0o/OrRkh8BiNTMd8I3w=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': 